In [12]:
import os
import cv2
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm

# Image folders
image_dirs = [
    "/home/ubuntu/HAM10000/HAM10000_images_part_1",
    "/home/ubuntu/HAM10000/HAM10000_images_part_2"
]

metadata_path = "/home/ubuntu/HAM10000/HAM10000_metadata.csv"
output_dir = "/home/ubuntu/HAM10000/HAM10000_augmented"

os.makedirs(output_dir, exist_ok=True)

In [13]:
df = pd.read_csv(metadata_path)

image_ids = df['image_id'].values
labels = df['dx'].values

print("Total samples:", len(df))

Total samples: 10015


In [14]:
counter = Counter(labels)
print("Class distribution BEFORE:")
print(counter)

Class distribution BEFORE:
Counter({'nv': 6705, 'mel': 1113, 'bkl': 1099, 'bcc': 514, 'akiec': 327, 'vasc': 142, 'df': 115})


In [15]:
minority_classes = ['mel', 'bcc', 'akiec', 'df', 'vasc']

In [16]:
image_map = {}

for folder in image_dirs:
    for file in os.listdir(folder):
        if file.endswith(".jpg"):
            img_id = file.split(".")[0]
            image_map[img_id] = os.path.join(folder, file)

print("Total images mapped:", len(image_map))

Total images mapped: 10015


In [17]:
def augment_image(img):
    aug_images = []
    h, w = img.shape[:2]

    # Rotation
    for angle in [-20, 20]:
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
        aug_images.append(cv2.warpAffine(img, M, (w, h)))

    # Flip
    aug_images.append(cv2.flip(img, 1))

    # Zoom
    crop = img[int(0.1*h):int(0.9*h), int(0.1*w):int(0.9*w)]
    aug_images.append(cv2.resize(crop, (w, h)))

    # Blur
    aug_images.append(cv2.GaussianBlur(img, (3,3), 0))

    # Brightness
    aug_images.append(cv2.convertScaleAbs(img, alpha=1.1, beta=10))

    return aug_images

In [18]:
print("Processing images one by one...")

label_counter = Counter(labels)
print("Original distribution:", label_counter)

Processing images one by one...
Original distribution: Counter({'nv': 6705, 'mel': 1113, 'bkl': 1099, 'bcc': 514, 'akiec': 327, 'vasc': 142, 'df': 115})


In [19]:
target_size = 2000

In [20]:
new_records = []

from collections import Counter
current_counts = Counter(labels)

for img_id, label in tqdm(zip(image_ids, labels), total=len(image_ids)):

    img_path = image_map.get(img_id, None)
    if img_path is None:
        continue

    img = cv2.imread(img_path)

    # Save original
    new_records.append((img_id, label))

    # 🎯 Only augment minority classes
    if label in minority_classes:

        # 🔥 Dynamic target per class
        if label in ['df', 'vasc']:
            target_size = 1200
            max_aug_per_image = 8   # aggressive
        elif label in ['akiec', 'bkl']:
            target_size = 1300
            max_aug_per_image = 5
        else:  # mel, bcc
            target_size = 1500
            max_aug_per_image = 3

        if current_counts[label] >= target_size:
            continue

        needed = target_size - current_counts[label]

        # 🎯 smarter augmentation count
        num_aug = min(max_aug_per_image, needed)

        for i in range(num_aug):

            aug_imgs = augment_image(img)
            new_img = aug_imgs[np.random.randint(len(aug_imgs))]

            new_name = f"aug_{label}_{img_id}_{i}"
            save_path = os.path.join(output_dir, new_name + ".jpg")

            cv2.imwrite(save_path, new_img)

            new_records.append((new_name, label))
            current_counts[label] += 1

            # stop if target reached mid-loop
            if current_counts[label] >= target_size:
                break

100%|██████████| 10015/10015 [01:15<00:00, 132.51it/s]


In [21]:
final_df = pd.DataFrame(new_records, columns=["image_id", "dx"])
final_df.to_csv("/home/ubuntu/HAM10000/HAM10000_balanced.csv", index=False)

print("Done ✅")

Done ✅
